# Causal Inference in Practice
## Week 9 — Regression Discontinuity · Practice Notebook

> **Block III — Quasi-experimental designs**
>
> Just above vs. just below a cutoff is almost a coin flip — so the cutoff hands you a local experiment.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A cutoff is a local experiment

In a **regression discontinuity** design, treatment is decided by whether a *running variable* `R` crosses a known cutoff `c`. Far from `c`, treated and untreated units differ a lot. But right at `c`, a unit just above and one just below are nearly interchangeable — so the **jump in the outcome at the cutoff** is the causal effect, for units at the cutoff.

We'll simulate data where we *set* the true jump, so we can always check our estimate against the truth. We reuse the notebook's `RNG` and import `statsmodels` once here.

In [ ]:
import statsmodels.api as sm

def make_sharp(n=4000, c=0.0, tau=8.0, slope=4.0, curve=1.5, noise=3.0):
    """Sharp RD: D = 1 iff R >= c, with a KNOWN jump `tau` at c."""
    R = RNG.uniform(-1, 1, n)                 # running variable
    D = (R >= c).astype(float)                # sharp treatment
    mu = 30 + slope*R + curve*R**2            # smooth baseline (continuous at c)
    Y = mu + tau*D + RNG.normal(0, noise, n)  # jump of `tau` at the cutoff
    return pd.DataFrame({'R': R, 'D': D, 'Y': Y})

C, TAU = 0.0, 8.0          # cutoff and TRUE effect at the cutoff
sharp = make_sharp(c=C, tau=TAU)
print(sharp.head(3))
print(f'\nTrue jump at the cutoff: {TAU}')

### A picture first

Scatter the data and overlay a separate **local** linear fit on each side of the cutoff — fit only on points within a bandwidth `h` of `c`, the same window the estimator uses (RD is a *local* comparison, never a global curve). The vertical gap between the two fitted lines *at* `c` is exactly what RD estimates.

In [ ]:
def side_fit(df, c, side, h=0.5):
    """Local linear fit on one side, within bandwidth h of c."""
    d = df[(df['D'] == side) & (np.abs(df['R'] - c) <= h)]
    b = np.polyfit(d['R'], d['Y'], 1)        # local linear fit per side
    xs = np.linspace(c - h, c, 50) if side == 0 else np.linspace(c, c + h, 50)
    return xs, np.polyval(b, xs), np.polyval(b, c)

fig, ax = plt.subplots()
samp = sharp.sample(800, random_state=0)
ax.scatter(samp['R'], samp['Y'], s=6, alpha=0.3, color='#7F8694')
at_c = {}
for side, col in [(0.0, '#2F6DB5'), (1.0, '#2A9D8F')]:
    xs, ys, yc = side_fit(sharp, C, side)
    ax.plot(xs, ys, color=col, lw=2.5)
    at_c[side] = yc
ax.axvline(C, color='#C0504D', ls='--')
ax.set_xlabel('running variable R'); ax.set_ylabel('outcome Y')
ax.set_title('Sharp RD: the jump at the cutoff is the effect')
print(f'gap between the fitted lines at R=0 = {at_c[1.0] - at_c[0.0]:.2f}'
      f'  (should be about {TAU})')

## 2 · Estimate the jump by local linear regression

Keep only points within a **bandwidth** `h` of the cutoff and fit

$$Y = \beta_0 + \tau\,D + \gamma\,(R-c) + \delta\,D\,(R-c) + \varepsilon.$$

The coefficient `τ` on the treatment dummy `D` is the jump at `c`. We print the estimate next to the truth and `assert` we recovered it.

In [ ]:
def rd_estimate(df, c=0.0, h=0.5):
    """Local linear RD: returns the estimated jump at the cutoff."""
    d = df[np.abs(df['R'] - c) <= h]
    xc = d['R'].values - c
    Z = np.column_stack([np.ones(len(d)), d['D'].values, xc, d['D'].values*xc])
    return sm.OLS(d['Y'].values, Z).fit().params[1]   # coef on D

est = rd_estimate(sharp, c=C, h=0.5)
print(f'local linear RD estimate = {est:.3f}   (truth = {TAU})')
assert abs(est - TAU) < 1.0, 'should recover the known jump'
print('Recovered the known jump within tolerance.')

### 🔧 Exercise 2.1 — the naive within-window difference

Before trusting the regression, build intuition with the crudest possible estimator: inside a *narrow* window around `c`, just take the **mean of `Y` above minus the mean below**. With a small enough window there is little curvature, so this simple difference should also land near `TAU`.

Fill in the `# TODO`s. The skeleton runs as-is (it uses `...`), so you can execute it before solving.

In [ ]:
def naive_jump(df, c=0.0, h=0.1):
    d = df[np.abs(df['R'] - c) <= h]
    above = ...   # TODO: mean of Y where D == 1 inside the window
    below = ...   # TODO: mean of Y where D == 0 inside the window
    return above, below

# Once filled in, this will print two numbers whose difference ≈ TAU.
naive_jump(sharp, c=C, h=0.1)

### ✅ Solution 2.1

In [ ]:
def naive_jump(df, c=0.0, h=0.1):
    d = df[np.abs(df['R'] - c) <= h]
    above = d.loc[d['D'] == 1, 'Y'].mean()
    below = d.loc[d['D'] == 0, 'Y'].mean()
    return above - below

nj = naive_jump(sharp, c=C, h=0.1)
print(f'naive within-window difference = {nj:.3f}   (truth = {TAU})')
assert abs(nj - TAU) < 1.5, 'narrow-window difference should be near the truth'

## 3 · Bandwidth sensitivity

Choosing `h` is a **bias–variance tradeoff**. A *narrow* `h` has little bias from curvature but is noisy; a *wide* `h` reaches into the curved baseline and biases the jump. A credible RD is **stable** across a sensible range. We estimate across several bandwidths and plot the result.

In [ ]:
hs = [0.10, 0.15, 0.20, 0.30, 0.50, 0.75, 1.00]
ests = [rd_estimate(sharp, c=C, h=h) for h in hs]
for h, e in zip(hs, ests):
    print(f'h = {h:>4}:  est = {e:.3f}')

fig, ax = plt.subplots()
ax.plot(hs, ests, 'o-', color='#2F6DB5')
ax.axhline(TAU, color='#C0504D', ls='--', label=f'truth = {TAU}')
ax.set_xlabel('bandwidth h'); ax.set_ylabel('estimated jump')
ax.set_title('Bandwidth sensitivity'); ax.legend()

# The mid-range estimates should cluster near the truth.
mid = [e for h, e in zip(hs, ests) if h <= 0.5]
assert abs(np.mean(mid) - TAU) < 1.0, 'small/medium-h estimates should be near truth'
print('\nMid-range bandwidths agree on the truth; watch the drift as h grows.')

## 4 · Placebo cutoffs — a falsification test

At a **fake** cutoff where no treatment actually changes, a valid RD estimator should return *approximately zero*. To make sure there is genuinely no jump, we restrict to **one side** of the real cutoff (all-control or all-treated) and invent a placebo cutoff inside it. A near-zero estimate is reassuring; a large one means the method is reading curvature, not treatment.

In [ ]:
def placebo_estimate(df, real_c, fake_c, h=0.3):
    # stay entirely on one side of the real cutoff -> no real treatment jump
    side = df[df['D'] == (1.0 if fake_c > real_c else 0.0)].copy()
    d = side[np.abs(side['R'] - fake_c) <= h]
    xc = d['R'].values - fake_c
    fakeD = (d['R'].values >= fake_c).astype(float)
    Z = np.column_stack([np.ones(len(d)), fakeD, xc, fakeD*xc])
    return sm.OLS(d['Y'].values, Z).fit().params[1]

for fc in [-0.5, 0.5]:
    pe = placebo_estimate(sharp, C, fc, h=0.3)
    print(f'placebo cutoff {fc:+.1f}:  est = {pe:+.3f}   (expect ~0)')
    assert abs(pe) < 2.0, 'placebo cutoff should give ~0 effect'
print('Placebo cutoffs return ~0 — the estimator is not inventing jumps.')

## 5 · Donut-hole robustness

If the effect were really driven by suspicious **heaping or manipulation right at the cutoff**, dropping a tiny window around `c` would change the answer. We re-estimate after removing units with `|R − c| < δ` for a few `δ` and confirm the estimate barely moves.

In [ ]:
def donut_estimate(df, c=0.0, h=0.5, delta=0.05):
    d = df[(np.abs(df['R'] - c) <= h) & (np.abs(df['R'] - c) >= delta)]
    xc = d['R'].values - c
    Z = np.column_stack([np.ones(len(d)), d['D'].values, xc, d['D'].values*xc])
    return sm.OLS(d['Y'].values, Z).fit().params[1]

for delta in [0.0, 0.03, 0.05, 0.10]:
    de = donut_estimate(sharp, c=C, h=0.5, delta=delta)
    print(f'donut delta = {delta:>4}:  est = {de:.3f}')
assert abs(donut_estimate(sharp, c=C, h=0.5, delta=0.05) - TAU) < 1.0
print('Estimate is stable when the hole is removed — no heaping artifact.')

### 🔧 Exercise 5.1 — a McCrary-style density check

Manipulation of the running variable shows up as a **jump in its density at the cutoff**: many more units land *just eligible* than *just ineligible*. Build a quick check: bin `R`, then compare the average bin count just below `c` to the average just above. For our clean simulation the ratio should be ≈ 1.

Fill in the `# TODO`s; the skeleton runs as-is.

In [ ]:
def density_ratio(R, c=0.0, window=0.2, bins=40):
    counts, edges = np.histogram(R, bins=bins)
    ctr = 0.5*(edges[:-1] + edges[1:])
    below = ...   # TODO: mean count for centers in [c-window, c)
    above = ...   # TODO: mean count for centers in [c, c+window)
    return below, above

density_ratio(sharp['R'].values, c=C)

### ✅ Solution 5.1

In [ ]:
def density_ratio(R, c=0.0, window=0.2, bins=40):
    counts, edges = np.histogram(R, bins=bins)
    ctr = 0.5*(edges[:-1] + edges[1:])
    below = counts[(ctr < c) & (ctr >= c - window)].mean()
    above = counts[(ctr >= c) & (ctr < c + window)].mean()
    return above / below

ratio = density_ratio(sharp['R'].values, c=C)
print(f'density ratio (above / below) = {ratio:.2f}   (~1 ⇒ no manipulation)')
assert 0.7 < ratio < 1.3, 'clean simulation should have a smooth density at c'

# Contrast: a manipulated running variable where units sort just above c.
Rman = sharp['R'].values.copy()
near_below = (Rman < C) & (Rman > C - 0.1)
movers = near_below & (RNG.random(len(Rman)) < 0.6)   # 60% jump the cutoff
Rman[movers] = C + RNG.uniform(0, 0.1, movers.sum())
ratio_man = density_ratio(Rman, c=C)
print(f'manipulated density ratio = {ratio_man:.2f}   (>> 1 ⇒ sorting detected)')
assert ratio_man > 1.3, 'manipulation should show up as a density spike above c'

## 6 · Fuzzy RD — IV at the cutoff

In a **fuzzy** design, crossing `c` does not flip treatment from 0 to 1 — it only raises the *probability* of treatment (say 0.2 → 0.7). The outcome jump then **understates** the per-treated effect, because not everyone above `c` is actually treated.

The fix is the Wald/IV ratio at the cutoff:

$$\hat\tau_{\text{fuzzy}} = \frac{\text{jump in }Y\text{ at }c}{\text{jump in }P(D=1)\text{ at }c} = \frac{\text{reduced form}}{\text{first stage}}.$$

We simulate a fuzzy RD with a known per-treated effect and recover it.

In [ ]:
def make_fuzzy(n=8000, c=0.0, tau=8.0, slope=4.0, noise=3.0,
               p_below=0.15, p_above=0.75):
    """Fuzzy RD: crossing c raises P(treated) from p_below to p_above."""
    R = RNG.uniform(-1, 1, n)
    above = (R >= c).astype(float)             # the instrument: eligible side
    prob = np.where(above == 1, p_above, p_below)
    D = RNG.binomial(1, prob).astype(float)    # actual treatment (probabilistic)
    Y = 30 + slope*R + tau*D + RNG.normal(0, noise, n)
    return pd.DataFrame({'R': R, 'D': D, 'Y': Y, 'above': above})

fuzzy = make_fuzzy(c=C, tau=TAU)
print(fuzzy.groupby('above')['D'].mean().rename('P(treated)'))

In [ ]:
def fuzzy_rd(df, c=0.0, h=0.5):
    """Wald ratio at the cutoff: (jump in Y) / (jump in P(D=1))."""
    d = df[np.abs(df['R'] - c) <= h]
    xc = d['R'].values - c
    Z = np.column_stack([np.ones(len(d)), d['above'].values, xc, d['above'].values*xc])
    reduced_form = sm.OLS(d['Y'].values, Z).fit().params[1]   # jump in Y
    first_stage  = sm.OLS(d['D'].values, Z).fit().params[1]   # jump in P(D)
    return reduced_form / first_stage, reduced_form, first_stage

late, rf, fs = fuzzy_rd(fuzzy, c=C, h=0.5)
print(f'reduced form (jump in Y)      = {rf:.3f}')
print(f'first stage  (jump in P(D=1)) = {fs:.3f}')
print(f'fuzzy RD = reduced / first    = {late:.3f}   (truth = {TAU})')
assert abs(late - TAU) < 1.5, 'Wald ratio should recover the per-treated effect'
print('\nDividing by the first stage restores the per-treated effect — IV at the cutoff.')

### 🔧 Exercise 6.1 — forgetting to scale

A common mistake is to report the **reduced form** (the raw outcome jump) as if it were the treatment effect, forgetting that only part of the above-cutoff group is treated. Compute the reduced form alone and show it is well *below* `TAU`; then confirm that dividing by the first stage fixes it.

Fill in the `# TODO`.

In [ ]:
late2, rf2, fs2 = fuzzy_rd(fuzzy, c=C, h=0.4)
naive_fuzzy = ...   # TODO: the reduced form alone (the WRONG answer)
# print('reduced form only:', round(naive_fuzzy, 3), ' vs truth', TAU)

### ✅ Solution 6.1

In [ ]:
late2, rf2, fs2 = fuzzy_rd(fuzzy, c=C, h=0.4)
naive_fuzzy = rf2                      # reporting the reduced form alone
print(f'reduced form only = {naive_fuzzy:.3f}   (truth = {TAU}) -> too small')
print(f'scaled (Wald)     = {late2:.3f}   (truth = {TAU}) -> correct')
assert naive_fuzzy < TAU - 1.0, 'the raw jump understates the per-treated effect'
assert abs(late2 - TAU) < 1.5, 'scaling by the first stage recovers the truth'

## Wrap-up & self-check

- A **cutoff** on a running variable creates a local experiment: just above vs. just below is almost a coin flip.
- **Sharp RD** identifies the ATE at the cutoff; **fuzzy RD** identifies a LATE via the **Wald ratio** (reduced form ÷ first stage).
- The identifying assumption is **continuity** at `c` — and you estimate the jump with **local linear regression**, never a global high-order polynomial.
- Choosing the **bandwidth** trades bias against variance; report the estimate across a range.
- Earn the design with **validity checks**: a McCrary density test for manipulation, **placebo cutoffs** (≈0), and a **donut-hole** check (stable).

**You're ready for Week 10** if you can state what each design identifies, recover a known jump in code, and explain what each check would catch. Next week: difference-in-differences trades a cutoff in space for a break in time.